# Public-source note

Refines selected ATAC ChromHMM states (A7-A19) using state-associated ATAC peak summits and removes intervals also supported by non-associated conditions. Produces `ATAC_23_filtered_segments.bed`. Outputs were stripped; path cells were sanitized.

In [ ]:
import os, shlex, subprocess, sys
import pandas as pd

import os, shlex, subprocess, sys
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
atac_peak_dir="/path/to/CEBPB_seq/atac/all_samples_dedup_macs2"
reordered_chromhmm_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_atac/numstates_23_reordered/"
merged_seg_path= Path(reordered_chromhmm_dir) / "ATAC_23_segments.bed"
emissions_path = Path(reordered_chromhmm_dir) / "emissions_23.txt"
states_beds_dir = Path(reordered_chromhmm_dir) / "states_beds"
state_summit_overlap_dir = Path(reordered_chromhmm_dir) / "states_peaks_summit_intersect"
state_summit_overlap_filtered_dir = Path(reordered_chromhmm_dir) / "state_summit_overlap_filtered"
merged_filtered_seg_path= Path(state_summit_overlap_filtered_dir) / "ATAC_23_filtered_segments.bed"

state_filtered_summit_files_dir = Path(state_summit_overlap_filtered_dir) / "state_filtered_summit_files"




In [ ]:
emissions_df = pd.read_csv(emissions_path, sep="\t", index_col=0)
emissions_df

In [ ]:
states=list(emissions_df.index)
print(states)

In [ ]:
# peak_summits_dict={
#     'MEF':'MEF_ATAC_mm10_macs2_summits.bed',
#     'MEF_d2_OSKM':'MEF_d2_OSKM_ATAC_L002_mm10_macs2_summits.bed',
#     'MEF_d2_OSKM_mCEBPAflag':'MEF_d2_OSKM_mCEBPAflag_ATAC_mm10_macs2_summits.bed', 
#     'MEF_d2_OSKM_mCEBPBflag':'MEF_d2_OSKM_mCEBPBflag_ATAC_mm10_macs2_summits.bed', 
#     'MEF_d2_OSKM_hCEBPDflag':'MEF_d2_OSKM_hCEBPDflag_ATAC_mm10_macs2_summits.bed', 
#     'MEF_d2_mCEBPAflag':'MEF_d2_mCEBPAflag_ATAC_mm10_macs2_summits.bed',
#     'MEF_d2_mCEBPBflag' :'MEF_d2_mCEBPBflag_ATAC_mm10_macs2_summits.bed' ,
    
#        'NSC':'NSC_ATAC_S23_L005_mm10_macs2_summits.bed',
#     'NSC_d2_OSKM':'NSC_d2_OSKM_ATAC_mm10_macs2_summits.bed',
#  'NSC_d2_OSKM_mCEBPAflag': 'NSC_d2_OSKM_mCEBPAflag_ATAC_mm10_macs2_summits.bed',
#   'NSC_d2_mCEBPBflag' :'NSC_d2_mCEBPBflag_ATAC_mm10_macs2_summits.bed', 
#   'NSC_d2_mCEBPAflag':'NSC_d2_mCEBPAflag_ATAC_S15_mm10_macs2_summits.bed',
#    'NSC_d2_OSKM_mCEBPBflag':'NSC_d2_OSKM_mCEBPBflag_ATAC_mm10_macs2_summits.bed',

#     'ESC':'ESC_ATAC-Seq_mm10_sorted_dedup_macs2_summits.bed',
#     'ESC_d1_hCEBPA':'ESC_d1_hCEBPA_ATAC_mm10_macs2_summits.bed',
#     'ESC_d2_hCEBPA' :'ESC_d2_hCEBPA_ATAC_mm10_macs2_summits.bed', 

#         'Bcell':'Bcell_Graf_ATAC_2_merged_R2_mm10_sorted_dedup_macs2_summits.bed',
#     'Bcell_Cebpa18h':'Bcell_Cebpa18h_Graf_ATAC_merged_R1_mm10_sorted_dedup_macs2_summits.bed',
#     'Bcell_Cebpa18h_OSKMd1' :'Bcell_Cebpa18h_OSKMd1_Graf_ATAC_merged_mm10_sorted_dedup_macs2_summits.bed',
#     'Bcell_Cebpa18h_OSKMd2' :'Bcell_Cebpa18h_OSKMd2_Graf_ATAC_merged_mm10_sorted_dedup_macs2_summits.bed'
 
# }
# len(peak_summits_dict.keys()),len(peak_summits_dict.values())

In [ ]:
peak_summits_dict={
    'MEF':'MEF_ATAC_mm10_sorted_dedup_macs2_summits.bed',
    'MEF_d2_OSKM':'MEF_d2_OSKM_ATAC__mm10_sorted_dedup_macs2_summits.bed',
    'MEF_d2_OSKM_mCEBPAflag':'MEF_d2_OSKM_mCEBPAflag_ATAC_mm10_sorted_dedup_macs2_summits.bed', 
    'MEF_d2_OSKM_mCEBPBflag':'MEF_d2_OSKM_mCEBPBflag_ATAC_mm10_sorted_dedup_macs2_summits.bed', 
    'MEF_d2_OSKM_hCEBPDflag':'MEF_d2_OSKM_hCEBPDflag_ATAC_dedup_macs2_summits.bed', 
    'MEF_d2_mCEBPAflag':'MEF_d2_mCEBPAflag_ATAC_mm10_sorted_dedup_macs2_summits.bed',
    'MEF_d2_mCEBPBflag' :'MEF_d2_mCEBPBflag_ATAC_mm10_sorted_dedup_macs2_summits.bed' ,
    
       'NSC':'NSC_ATAC_mm10_sorted_dedup_macs2_summits.bed',
    'NSC_d2_OSKM':'NSC_d2_OSKM_ATAC_mm10_sorted_dedup_macs2_summits.bed',
 'NSC_d2_OSKM_mCEBPAflag': 'NSC_d2_OSKM_mCEBPAflag_ATAC_mm10_sorted_dedup_macs2_summits.bed',
  'NSC_d2_mCEBPBflag' :'NSC_d2_mCEBPBflag_ATAC_mm10_sorted_dedup_macs2_summits.bed', 
  'NSC_d2_mCEBPAflag':'NSC_d2_mCEBPAflag_ATAC_mm10_sorted_dedup_macs2_summits.bed',
   'NSC_d2_OSKM_mCEBPBflag':'NSC_d2_OSKM_mCEBPBflag_ATAC_mm10_sorted_dedup_macs2_summits.bed',

    'ESC':'ESC_ATAC-Seq_mm10_sorted_dedup_macs2_summits.bed',
    'ESC_d1_hCEBPA':'ESC_d1_hCEBPA_ATAC_mm10_sorted_dedup_macs2_summits.bed',
    'ESC_d2_hCEBPA' :'ESC_d2_hCEBPA_ATAC_sorted_dedup_macs2_summits.bed', 

        'Bcell':'Bcell_Graf_ATAC_2_merged_R2_mm10_sorted_dedup_macs2_summits.bed',
    'Bcell_Cebpa18h':'Bcell_Cebpa18h_Graf_ATAC_merged_R1_mm10_sorted_dedup_macs2_summits.bed',
    'Bcell_Cebpa18h_OSKMd1' :'Bcell_Cebpa18h_OSKMd1_Graf_ATAC_merged_mm10_sorted_dedup_macs2_summits.bed',
    'Bcell_Cebpa18h_OSKMd2' :'Bcell_Cebpa18h_OSKMd2_Graf_ATAC_merged_mm10_sorted_dedup_macs2_summits.bed'
 
}
len(peak_summits_dict.keys()),len(peak_summits_dict.values())

## IntersectBed all states with all atac summits files

In [ ]:
for state in states:
    state_bed_path= Path(states_beds_dir) / str(state+".bed")
    print(state_bed_path)
    for key, value in peak_summits_dict.items():
        print(key, value)
        peak_summits_path=atac_peak_dir+'/'+value
        output_bed=state_summit_overlap_dir/ str(state+"_intersectBed_"+key+".bed")
        print(output_bed)
        cmd = [
        "intersectBed",
        "-a", str(peak_summits_path),
        "-b", str(state_bed_path),
        # "-wa"  # write entries from A that overlap B
        "-wb",  # uncomment if you also want entries from B
        ]
    
        with open(output_bed, "w") as out:
            subprocess.run(cmd, stdout=out, check=True)
            print(f"Intersection written to {output_bed}")
    #     break
    # break

In [ ]:
peak_summits_dict.keys()

In [ ]:
state_sample_dict={'A1':[], 
 'A2':[], 
 'A3':[], 
 'A4':[], 
 'A5':[], 
 'A6':[], 
 'A7':['MEF', 'MEF_d2_OSKM'], 
 
 'A8':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag'], 
 'A9':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag'], 
 
 
 
 'A10':[ 'NSC', 'NSC_d2_OSKM'], 
 
 'A11':['NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag'], 
 
 'A12':['NSC', 'NSC_d2_OSKM', 'NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag'],
 
 'A13':['ESC'], 
 
 'A14':['ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'], 
 
 'A15':['ESC','ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'], 
 
 'A16':['Bcell'], 
 
 'A17':['Bcell_Cebpa18h_OSKMd1'],
 
 'A18':['Bcell', 'Bcell_Cebpa18h', 'Bcell_Cebpa18h_OSKMd1', 'Bcell_Cebpa18h_OSKMd2'], 
 
 'A19':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag',
        'NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag',
        'ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'    ],
 
 'A20':[], 
 'A21':[], 
 'A22':[], 
 'A23':[]
}




 

In [ ]:
print(states)

## Filter intersectBed files by non-state samples

In [ ]:
## Remove rows of states that overlap with samples NOT belonging to this stateimport pandas as pd

# Only process selected states
keep_states = {"A7","A8","A9", 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19'}

for state in states:
    # Skip states that are not in keep_states
    if state not in keep_states:
        continue

    # Samples that belong to this state
    state_samples = state_sample_dict[state]

    for state_sample in state_samples:

        # Path to the original intersectBed file for this state and sample
        sample_path = state_summit_overlap_dir / f"{state}_intersectBed_{state_sample}.bed"

        # Path where the filtered result will be saved
        sample_filtered_path = state_summit_overlap_filtered_dir / f"{state}_intersectBed_{state_sample}.bed"

        # Load the sample intersect file
        state_sample_df = pd.read_csv(sample_path, sep="\t", header=None)

        # Start with all rows, progressively remove overlaps
        filtered_df = state_sample_df.copy()

        print(f"\n\n*********************************************************************************\n\n")
        print(f"\nState {state} | Sample {state_sample} | Starting rows: {len(filtered_df)}")

        # Loop over all samples in peak_summits_dict
        # Remove rows that overlap with samples NOT belonging to this state
        for non_sample in peak_summits_dict.keys():

            # Skip samples that belong to this state
            if non_sample in state_samples:
                continue

            # Load intersect file for the non-state sample
            non_sample_path = state_summit_overlap_dir / f"{state}_intersectBed_{non_sample}.bed"
            state_non_sample_df = pd.read_csv(non_sample_path, sep="\t", header=None)

            # Reset index and create a temporary row id
            # This ensures we can safely track rows after merge
            tmp = (
                filtered_df
                .reset_index(drop=False)
                .rename(columns={"index": "_rid"})
            )

            # Keep only the comparison columns (6,7,8)
            # Drop duplicates to speed up merge
            non_sub = state_non_sample_df[[6, 7, 8]].drop_duplicates()

            # Left merge to identify matching rows
            merged = tmp.merge(
                non_sub,
                on=[6, 7, 8],
                how="left",
                indicator=True
            )

            # Keep only rows that did NOT match ("left_only")
            keep_rids = merged.loc[merged["_merge"] == "left_only", "_rid"]

            # Filter the original dataframe using row ids
            filtered_df = filtered_df.loc[keep_rids].copy()
            print(f"After removing overlaps with {non_sample}: {len(filtered_df)}")

        print(f"Final rows after removing overlaps: {len(filtered_df)}")

        # Save filtered file
        filtered_df.to_csv(
            sample_filtered_path,
            sep="\t",
            header=None,
            index=None
        )

        # break  # Uncomment if you want to process only one sample per state


## Get the filtered intersectBed files

In [ ]:
import pandas as pd
from pathlib import Path

# States to process
keep_states = {
    "A7","A8","A9","A10","A11","A12",
    "A13","A14","A15","A16","A17","A18","A19"
}

for state in states:

    # Skip unwanted states
    if state not in keep_states:
        continue

    print(f"\nState: {state}")

    # Load the original state bed (used here only for reporting row counts)
    state_bed_path = Path(states_beds_dir) / f"{state}.bed"
    state_df = pd.read_csv(state_bed_path, sep="\t", header=None)

    # Samples that belong to this state
    state_samples = state_sample_dict[state]
    print(f"Samples: {state_samples}")

    # Collect per sample filtered DataFrames to merge
    dfs_to_merge = []

    for state_sample in state_samples:

        # Input: filtered intersectBed output for this (state, sample)
        sample_filtered_path = (
            state_summit_overlap_filtered_dir /
            f"{state}_intersectBed_{state_sample}.bed"
        )

        # Load full table, keep all columns
        df = pd.read_csv(sample_filtered_path, sep="\t", header=None)

        # Optional: remove exact duplicate rows within the same file
        dfs_to_merge.append(df.drop_duplicates())

    # Merge all samples for this state
    if not dfs_to_merge:
        continue

    filtered_merged_df = pd.concat(dfs_to_merge, ignore_index=True)

    print(f"Original state rows: {len(state_df)}")
    print(f"Merged rows before dedup: {len(filtered_merged_df)}")

    # Drop duplicates using only these columns as the key
    # This keeps the first full row encountered for each unique key
    dedup_key = [5, 6, 7, 8]
    filtered_merged_df = filtered_merged_df.drop_duplicates(subset=dedup_key, keep="first")

    print(f"Merged rows after dedup by {dedup_key}: {len(filtered_merged_df)}")

    # Save 1: the dedup key columns (5,6,7,8)
    out_key_path = state_summit_overlap_filtered_dir / f"{state}_filtered.bed"
    filtered_merged_df[dedup_key].to_csv(out_key_path, sep="\t", header=False, index=False)

    # Save 2: the first 4 columns (0,1,2,3) as a BED for summits or regions
    out_bed_path = state_filtered_summit_files_dir / f"{state}_filtered.bed"
    filtered_merged_df[[0, 1, 2, 3]].to_csv(out_bed_path, sep="\t", header=False, index=False)

    # Remove this break to process all states
    # break


## Get_ATAC_23_filtered_segments

In [ ]:
import pandas as pd
from pathlib import Path

# States to process
keep_states = {
    "A7","A8","A9","A10","A11","A12",
    "A13","A14","A15","A16","A17","A18","A19"
}


# Collect per sample state DataFrames to merge
dfs_to_merge = []

for state in states:

    # read the un filtered bed file
    if state not in keep_states:
        print(f"\nState: {state}")
        # Load the original state bed (used here only for reporting row counts)
        state_bed_path = states_beds_dir / f"{state}.bed"
        state_df = pd.read_csv(state_bed_path, sep="\t", header=None)
        print(len(state_df))
        dfs_to_merge.append(state_df.drop_duplicates())
        # continue
    # read the filtered bed file
    else:
        # Load the original state bed (used here only for reporting row counts)
        state_bed_path = state_summit_overlap_filtered_dir / f"{state}_filtered.bed"
        state_df = pd.read_csv(state_bed_path, sep="\t", header=None)
        print(len(state_df))
        dfs_to_merge.append(state_df.drop_duplicates())
filtered_merged_df = pd.concat(dfs_to_merge, ignore_index=True)
filtered_merged_df.to_csv(merged_filtered_seg_path, sep="\t", header=False, index=False)



In [ ]:
filtered_merged_df.head()

In [ ]:
set(filtered_merged_df[3])

In [ ]:
dict <- list(
  H1 = c("H3K4me3", "H3K9ac"),
  H2 = c("H3K4me1")
)


In [ ]:
 'A7':['MEF', 'MEF_d2_OSKM'], 
 
 'A8':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag'], 
 'A9':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag'], 
 
 
 
 'A10':[ 'NSC', 'NSC_d2_OSKM'], 
 
 'A11':['NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag'], 
 
 'A12':['NSC', 'NSC_d2_OSKM', 'NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag'],
 
 'A13':['ESC'], 
 
 'A14':['ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'], 
 
 'A15':['ESC','ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'], 
 
 'A16':['Bcell'], 
 
 'A17':['Bcell_Cebpa18h_OSKMd1'],
 
 'A18':['Bcell', 'Bcell_Cebpa18h', 'Bcell_Cebpa18h_OSKMd1', 'Bcell_Cebpa18h_OSKMd2'], 
 
 'A19':['MEF_d2_OSKM_mCEBPAflag', 'MEF_d2_OSKM_mCEBPBflag', 'MEF_d2_OSKM_hCEBPDflag', 'MEF_d2_mCEBPAflag', 'MEF_d2_mCEBPBflag',
        'NSC_d2_OSKM_mCEBPAflag', 'NSC_d2_mCEBPBflag', 'NSC_d2_mCEBPAflag', 'NSC_d2_OSKM_mCEBPBflag',
        'ESC_d1_hCEBPA', 'ESC_d2_hCEBPA'    ],

In [ ]:
program_order <- c(
  "MEF",
  "MEF_d2_OSKM",
  "MEF_d2_OSKM_mCEBPA",
  "MEF_d2_OSKM_mCEBPB",
  "MEF_d2_mCEBPA",
  'MEF_d2_mCEBPB' ,
  'NSC' ,
'NSC_d2_mCEBPA' ,
'NSC_d2_mCEBPB' ,
'NSC_d2_OSKM' ,
'NSC_d2_OSKM_mCEBPA' ,
'NSC_d2_OSKM_mCEBPB' ,
  'ESC' ,
'ESC_d1_hCEBPA' ,
'ESC_d2_hCEBPA'  ,
'Bcell_Graf' ,
"Bcell_Graf_Cebpa18h",
'Bcell_Graf_OKSMd1' ,
'Bcell_Graf_OKSMd2'
)
